# Paired clean/noisy retrieval on the existing Titans and TTT-Linear checkpoints

Fixed three distractors per noisy context. At distance 0, the distractors share the target segment.
Each trial reuses the same key, target value and four candidates across conditions and distances.
The window baseline sees the query. The full-context check sees the query and all context.
The rank metric is teacher-forced candidate scoring. Token argmax is also teacher forced.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "titans-pytorch==0.6.0", "transformers", "datasets", "accelerate", "matplotlib"])

In [ ]:
import os, time, math, json, re, random, glob, html as _html
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

DEBUG = False
KINDS = ["titans", "ttt_linear"]       # loại nào có checkpoint thì được so sánh
SEED = 0

CKPT_PATHS = {}
for _k in KINDS:
    _env = os.environ.get(f"CKPT_{_k.upper()}")
    _c = [_env] if _env else sorted(glob.glob(f"/kaggle/input/**/{_k}_vi_memory.pt", recursive=True)) + \
                             [f"/kaggle/working/ckpt_{_k}/{_k}_vi_memory.pt"]
    _c = [p for p in _c if p and os.path.exists(p)]
    if _c:
        CKPT_PATHS[_k] = _c[0]
print("Checkpoint tìm thấy:", CKPT_PATHS)
assert {"titans", "ttt_linear"} <= CKPT_PATHS.keys(), "Cần ít nhất hai checkpoint để so sánh (xem hướng dẫn ở đầu notebook)."

_blob0 = torch.load(next(iter(CKPT_PATHS.values())), map_location="cpu")
_c0 = _blob0["cfg"]
MODEL_NAME, SEG_LEN, N_SEGS = _c0["model_name"], _c0["seg_len"], _c0["n_segs"]
MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD, MEM_DEPTH = _c0["mem_chunk"], _c0["mem_heads"], _c0["mem_dim_head"], _c0["mem_depth"]
USE_MOMENTUM = _c0.get("use_momentum", True)
TITANS_INNER_LR_CAP = 1.0  # Default in older Titans checkpoints
HOOK_LAYER = _c0["layer"]
for _k, _p in CKPT_PATHS.items():
    _c = torch.load(_p, map_location="cpu")["cfg"]
    if _k == "titans":
        TITANS_INNER_LR_CAP = _c.get("titans_inner_lr_cap", 1.0)
    for key in ["model_name", "seg_len", "n_segs", "mem_chunk", "mem_heads", "mem_dim_head", "layer"]:
        assert _c.get(key) == _c0.get(key), f"Cấu hình khác nhau giữa các checkpoint ở '{key}': {_c.get(key)} vs {_c0.get(key)}"
N_TRAIN_DOCS = int(os.environ.get("N_TRAIN_DOCS", 6000))
N_EVAL_DOCS = int(os.environ.get("N_EVAL_DOCS", 24))
EXCLUDE_TITLE_REGEX = r"vật lý|cơ học|nhiệt động|điện từ|lượng tử|tương đối|newton|động lực học|năng lượng"
BATCH_DOCS, TBPTT = 2, 2
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/working/data_vi.pt")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DEBUG:", DEBUG, "| cấu hình chung:", dict(seg=SEG_LEN, n_segs=N_SEGS, chunk=MEM_CHUNK, heads=MEM_HEADS))
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s
    display = print

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen2Config, Qwen2ForCausalLM

if DEBUG:
    torch.manual_seed(1234)     # backbone ngẫu nhiên phải giống nhau giữa các notebook DEBUG
    _cfg = Qwen2Config(vocab_size=500, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                       num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096)
    lm = Qwen2ForCausalLM(_cfg)
    def encode(t): return [b % 500 for b in t.encode("utf-8")]
    def decode(ids): return "".join(chr(32 + (int(i) % 90)) for i in ids)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _dt = torch.float32  # Avoid FP16 overflow in the frozen backbone on T4
    try:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=_dt)
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=_dt)
    def encode(t): return tok(t, add_special_tokens=False)["input_ids"]
    def decode(ids): return tok.decode(ids)

lm = lm.to(device).eval()
for p in lm.parameters():
    p.requires_grad_(False)
print("tham số backbone (đóng băng), triệu:", round(sum(p.numel() for p in lm.parameters()) / 1e6, 1))

In [ ]:

paths = sorted(glob.glob("/kaggle/input/**/data_vi.pt", recursive=True))
if not paths:
    raise FileNotFoundError("Mount data_vi.pt from the original Titans training output")
data = torch.load(paths[0], map_location="cpu", weights_only=True)
if tuple(data.shape) != (6024, 2048):
    raise ValueError(f"Unexpected tokenized corpus: {data.shape}")
eval_docs = data[-N_EVAL_DOCS:]


In [ ]:
# ---- TTT layer (đã kiểm tra số học: dual form == vòng lặp từng token, sai số ~1e-15) ----
# ==== TTT layer (Sun et al., "Learning to (Learn at Test Time)") dùng làm memory phụ ====
# Cài đặt riêng theo mô tả trong paper (không phải code chính thức):
#   loss      l(W; x) = 1/2 || f(theta_K x; W) - theta_V x ||^2           (Eq. 4, hệ số 1/2 gộp vào lr)
#   f(x; W)   = x + LN(f_res(x; W)),  f_res = Linear (TTT-Linear) hoac MLP 2 lop GELU, hidden = 4 * dim (TTT-MLP)
#   mini-batch TTT (b = block): gradient cua moi token trong block tinh tai W dau block (t' = t - mod(t, b));
#       W_t = W_t' - sum_{s=t'+1..t} eta_s * grad l(W_t'; x_s)             (mini-batch GD)
#   output    z_t = f(theta_Q x_t; W_t)                                    (Eq. 5)
#   eta(x)    = eta_base * sigmoid(theta_lr . x)  (eta_base = 1 Linear, 0.1 MLP);  W_0 hoc duoc (outer loop)
# Tinh toan bang "dual form": khong tao W_t rieng cho tung token, ket qua tuong duong ve mat toan hoc.
# Ghi chu: chia eta cho head_dim la lua chon de on dinh (theo thong le cua cac ban cai dat tham khao), khong co trong paper.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def ln_fwd(x, gamma, beta, eps=1e-6):
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta


def ln_l2_bwd(x, target, gamma, beta, eps=1e-6):
    # gradient theo x cua 1/2 || LN(x) - target ||^2 (dang dong kin)
    D = x.shape[-1]
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    std = torch.sqrt(var + eps)
    x_hat = (x - mu) / std
    y = gamma * x_hat + beta
    g = (y - target) * gamma
    return (D * g - g.sum(-1, keepdim=True) - x_hat * (g * x_hat).sum(-1, keepdim=True)) / (D * std)


def gelu_grad(x):
    return 0.5 * (1 + torch.erf(x / math.sqrt(2))) + x * torch.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)


def _lower_eta(eta, k):
    # eta: (B, nh, k) -> E[t, s] = eta_s voi s <= t
    return torch.tril(eta.unsqueeze(-2).expand(*eta.shape[:-1], k, k))


def ttt_linear_block(st, q, k, v, eta, ln_w, ln_b):
    # q, k, v: (B, nh, kk, f); eta: (B, nh, kk); st: W1 (B,nh,f,f), b1 (B,nh,1,f)
    W, b = st["W1"], st["b1"]
    kk = k.size(-2)
    Z1 = k @ W + b
    G = ln_l2_bwd(Z1, v - k, ln_w, ln_b)
    loss_tok = ((k + ln_fwd(Z1, ln_w, ln_b) - v) ** 2).mean(-1)          # (B, nh, kk) "surprise"
    E = _lower_eta(eta, kk)
    A = torch.tril(q @ k.transpose(-1, -2))
    Zb = q @ W - (E * A) @ G + (b - E @ G)
    out = q + ln_fwd(Zb, ln_w, ln_b)
    W_new = W - (eta.unsqueeze(-1) * k).transpose(-1, -2) @ G
    b_new = b - (eta.unsqueeze(-1) * G).sum(-2, keepdim=True)
    return out, {"W1": W_new, "b1": b_new}, loss_tok


def ttt_mlp_block(st, q, k, v, eta, ln_w, ln_b):
    W1, b1, W2, b2 = st["W1"], st["b1"], st["W2"], st["b2"]
    kk = k.size(-2)
    Z1 = k @ W1 + b1
    X2 = F.gelu(Z1)
    Z2 = X2 @ W2 + b2
    G2 = ln_l2_bwd(Z2, v - k, ln_w, ln_b)
    G1 = (G2 @ W2.transpose(-1, -2)) * gelu_grad(Z1)
    loss_tok = ((k + ln_fwd(Z2, ln_w, ln_b) - v) ** 2).mean(-1)
    E = _lower_eta(eta, kk)
    A1 = torch.tril(q @ k.transpose(-1, -2))
    Z1b = q @ W1 - (E * A1) @ G1 + (b1 - E @ G1)
    X2b = F.gelu(Z1b)
    A2 = torch.tril(X2b @ X2.transpose(-1, -2))
    Z2b = X2b @ W2 - (E * A2) @ G2 + (b2 - E @ G2)
    out = q + ln_fwd(Z2b, ln_w, ln_b)
    e = eta.unsqueeze(-1)
    new = {
        "W1": W1 - (e * k).transpose(-1, -2) @ G1, "b1": b1 - (e * G1).sum(-2, keepdim=True),
        "W2": W2 - (e * X2).transpose(-1, -2) @ G2, "b2": b2 - (e * G2).sum(-2, keepdim=True),
    }
    return out, new, loss_tok


def flat_state(state):
    return torch.cat([state[n].flatten(1) for n in sorted(state)], dim=-1)


class TTTMem(nn.Module):
    """Giao dien: forward(x:(B,T,dim), state, want_trace) -> (read:(B,T,dim), new_state, info)."""

    def __init__(self, dim, heads=4, head_dim=64, block=16, kind="linear", eta_base=None):
        super().__init__()
        assert kind in ("linear", "mlp")
        self.nh, self.f, self.b, self.kind = heads, head_dim, block, kind
        self.eta_base = eta_base if eta_base is not None else (1.0 if kind == "linear" else 0.1)
        inner = heads * head_dim
        self.norm = RMSNorm(dim)
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_lr = nn.Linear(dim, heads)
        nn.init.normal_(self.to_lr.weight, std=0.02)
        nn.init.zeros_(self.to_lr.bias)
        self.merge = nn.Linear(inner, dim, bias=False)
        self.ln_w = nn.Parameter(torch.ones(heads, 1, head_dim))
        self.ln_b = nn.Parameter(torch.zeros(heads, 1, head_dim))
        f = head_dim
        if kind == "linear":
            self.W1 = nn.Parameter(torch.randn(heads, f, f) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1"]
        else:
            h = 4 * f
            self.W1 = nn.Parameter(torch.randn(heads, f, h) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, h))
            self.W2 = nn.Parameter(torch.randn(heads, h, f) * 0.02)
            self.b2 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1", "W2", "b2"]

    def init_state(self, B):
        return {n: getattr(self, n).unsqueeze(0).expand(B, *getattr(self, n).shape).clone() for n in self.state_names}

    def forward(self, x, state=None, want_trace=False):
        B, T, _ = x.shape
        h = self.norm(x)
        sp = lambda t: t.view(B, T, self.nh, self.f).transpose(1, 2)
        q, k, v = sp(self.to_q(h)), sp(self.to_k(h)), sp(self.to_v(h))
        eta = self.eta_base * torch.sigmoid(self.to_lr(h)).transpose(1, 2) / self.f      # (B, nh, T)
        if state is None:
            state = self.init_state(B)
        block = ttt_linear_block if self.kind == "linear" else ttt_mlp_block
        outs, losses = [], []
        snaps = [flat_state(state).detach()] if want_trace else None
        for s in range(0, T, self.b):
            e = min(s + self.b, T)
            o, state, l = block(state, q[:, :, s:e], k[:, :, s:e], v[:, :, s:e], eta[:, :, s:e], self.ln_w, self.ln_b)
            outs.append(o)
            losses.append(l)
            if want_trace:
                snaps.append(flat_state(state).detach())
        z = torch.cat(outs, 2).transpose(1, 2).reshape(B, T, -1)
        read = self.merge(z)
        info = None
        if want_trace:
            info = dict(surprise=torch.cat(losses, 2).detach().float().mean((0, 1)),
                        lr=eta.detach().float().mean((0, 1)),
                        snaps=torch.stack(snaps, dim=1))
        return read, state, info

In [ ]:
from titans_pytorch import NeuralMemory, mem_state_detach

class MemLM(nn.Module):
    # Backbone đóng băng + một module memory gắn vào hidden state ở giữa mạng.
    # Cùng cách đọc ra (out, gate) cho mọi loại memory để so sánh công bằng.
    def __init__(self, lm, kind, layer=None):
        super().__init__()
        self.lm, self.kind = lm, kind
        d = lm.config.hidden_size
        if kind == "titans":
            # batch_size=MEM_CHUNK: sau mỗi chunk memory mới được cập nhật và chunk sau tính gradient theo trọng số MỚI
            # (mini-batch gradient descent của paper). Nếu để batch_size=None, thư viện luôn tính surprise theo trọng số
            # khởi tạo nên memory không bao giờ ảnh hưởng tới surprise.
            self.mem = NeuralMemory(dim=d, chunk_size=MEM_CHUNK, batch_size=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD,
                                    momentum=USE_MOMENTUM, default_step_transform_max_lr=TITANS_INNER_LR_CAP,
                                    default_model_kwargs=dict(depth=MEM_DEPTH, expansion_factor=4.0))
            # titans-pytorch tạo tham số khởi tạo memory theo head bằng `repeat` (view stride 0) -> Adam không ghi được.
            for i, p in enumerate(self.mem.memory_model_parameters):
                self.mem.memory_model_parameters[i] = nn.Parameter(p.detach().clone().contiguous())
        elif kind in ("ttt_linear", "ttt_mlp"):
            self.mem = TTTMem(d, MEM_HEADS, MEM_DIM_HEAD, MEM_CHUNK, "linear" if kind == "ttt_linear" else "mlp")
        else:
            raise ValueError(kind)
        self.out = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.layer = layer if layer is not None else lm.config.num_hidden_layers // 2
        self.state, self.enabled, self.trace = None, True, False
        self.calls, self.w_ref = [], None
        lm.model.layers[self.layer].register_forward_hook(self._hook)

    def reset(self):
        self.state = None

    def detach_state(self):
        if self.state is None:
            return
        if self.kind == "titans":
            self.state = mem_state_detach(self.state)
        else:
            self.state = {k: v.detach() for k, v in self.state.items()}

    def fast_params(self):
        if self.kind == "titans":
            return sum(v.numel() for k, v in self.state.weights.items() if k.startswith("model.weights"))
        return sum(v.numel() for v in self.state.values())

    def trainable_parameters(self):
        return list(self.mem.parameters()) + list(self.out.parameters()) + [self.gate]

    def _mem_call(self, h):
        if self.kind == "titans":
            r, st, sur = self.mem(h, state=self.state, return_surprises=True)
            info = None
            if self.trace:
                flat = torch.cat([v.flatten(2) for k, v in st.updates.items() if k.startswith("model.weights")], dim=-1)
                info = dict(surprise=sur[0].detach().float().mean((0, 1)), lr=sur[1].detach().float().mean((0, 1)),
                            snaps=flat.detach().float())
            return r, st, info
        return self.mem(h, state=self.state, want_trace=self.trace)

    def _hook(self, module, args, output):
        if not self.enabled:
            return output
        h = output[0] if isinstance(output, tuple) else output
        r, st, info = self._mem_call(h.float())
        self.state = st
        if self.trace and info is not None:
            with torch.no_grad():
                snaps = info["snaps"]                                   # (B', n+1, P), snaps[:, 0] = trạng thái trước lần gọi
                if self.w_ref is None:
                    self.w_ref = snaps[:, 0].clone()
                dabs = (snaps - self.w_ref[:, None]).norm(dim=-1)
                dist = (dabs / self.w_ref.norm(dim=-1, keepdim=True)).mean(0)
                step = (snaps[:, 1:] - snaps[:, :-1]).norm(dim=-1).mean(0)
                self.calls.append(dict(surprise=info["surprise"].cpu().numpy(), theta=info["lr"].cpu().numpy(),
                                       dist=dist.cpu().numpy(), dist_abs=dabs.mean(0).cpu().numpy(), step=step.cpu().numpy()))
        h2 = h + (self.gate * self.out(r)).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2

def build_model(kind):
    m = MemLM(lm, kind, HOOK_LAYER)
    m.mem.to(device); m.out.to(device); m.gate.data = m.gate.data.to(device)
    return m

In [ ]:
adapters = {}
for k, p in CKPT_PATHS.items():
    m = build_model(k)
    blob = torch.load(p, map_location=device)
    m.mem.load_state_dict(blob["mem"]); m.out.load_state_dict(blob["out"]); m.gate.data = blob["gate"].to(device)
    m.mem.eval(); m.out.eval()
    m.best_train_eval = blob.get("best"); m.train_steps = blob.get("step")
    adapters[k] = m

def use(kind):
    # chỉ bật hook của adapter được chọn (các adapter dùng chung một backbone)
    for k, m in adapters.items():
        m.enabled = (k == kind)
        m.reset()
    return adapters[kind]

PRETTY = {"titans": "Titans", "ttt_linear": "TTT-Linear", "ttt_mlp": "TTT-MLP"}
assert {k: m.train_steps for k, m in adapters.items()} == {"titans": 2665, "ttt_linear": 3000}
print({PRETTY[k]: f"{m.train_steps} bước, best eval loss {m.best_train_eval:.4f}" for k, m in adapters.items()})

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

NAMES = ["Zorvan", "Kelmar", "Tavris", "Brunex", "Ondira", "Veltro", "Mirkan", "Sadrel", "Yoruvi", "Pelnax"]
METALS = ["Vexa", "Lorin", "Quasa", "Dermik", "Faltor", "Nimbra", "Oskal", "Rhedia"]
DISTANCES = [0, 1, 2, 4, 7]
SEEDS = [2026, 2027, 2028, 2029]
TRIALS_PER_SEED = 25
MODELS = ["window", "titans", "ttt_linear", "full"]


def record(name, metal, value):
    return f"Theo ghi chép Delta, định luật {name} cho biết hợp kim {metal} có hệ số truyền nhiệt bằng {value} đơn vị.\n"


def make_trial(seed, trial, distance):
    rng = random.Random(seed * 1000 + trial)
    name, metal = rng.choice(NAMES), rng.choice(METALS)
    values = [f"{v // 100},{v % 100:02d}" for v in rng.sample(range(110, 1000), 4)]
    other_name = rng.choice([item for item in NAMES if item != name])
    other_metal = rng.choice([item for item in METALS if item != metal])
    unrelated = (rng.choice([item for item in NAMES if item not in (name, other_name)]),
                 rng.choice([item for item in METALS if item not in (metal, other_metal)]))
    target = encode(record(name, metal, values[0]))
    if len(target) > SEG_LEN:
        raise ValueError("Target fact exceeds the segment length")
    decoys = [encode(record(*key, value)) for key, value in
              zip([(name, other_metal), (other_name, metal), unrelated], values[1:])]
    rng.shuffle(decoys)
    fillers = []
    for _ in range(distance + 1):
        doc = eval_docs[rng.randrange(len(eval_docs))]
        offset = rng.randrange(len(doc) - SEG_LEN + 1)
        fillers.append(doc[offset:offset + SEG_LEN].tolist())
    clean = [tokens.copy() for tokens in fillers]
    noisy = [tokens.copy() for tokens in fillers]
    clean[0][:len(target)] = target
    noisy[0][:len(target)] = target
    offsets = [len(target)] + [0] * distance
    for i, decoy in enumerate(decoys):
        seg = distance if i == 0 else max(0, distance - (i - 1))
        start = offsets[seg]
        stop = start + len(decoy)
        if stop > SEG_LEN:
            raise ValueError(f"Segment {seg} cannot contain all records without truncation")
        noisy[seg][start:stop] = decoy
        offsets[seg] = stop
    prompt = f"Câu hỏi: Theo định luật {name}, hợp kim {metal} có hệ số truyền nhiệt bao nhiêu? Trả lời:"
    prefix = encode(prompt)
    candidates = []
    for value in values:
        whole = encode(prompt + " " + value)
        if whole[:len(prefix)] != prefix or len(whole) <= len(prefix):
            raise ValueError("Tokenizer changed the prompt at the answer boundary")
        candidates.append(whole[len(prefix):])
    if len({len(ids) for ids in candidates}) != 1:
        raise ValueError("Answer candidates must have equal token counts for sequence scoring")
    return (torch.tensor(clean, device=device), torch.tensor(noisy, device=device),
            torch.tensor(prefix, device=device), candidates, values)


@torch.no_grad()
def score_candidates(mode, context, prompt, candidates):
    for adapter in adapters.values():
        adapter.enabled = False
        adapter.reset()
    adapter = adapters.get(mode)
    if adapter is not None:
        adapter.enabled = True
        for segment in context:
            lm(input_ids=segment[None], use_cache=False, logits_to_keep=1)
        prefix_state = adapter.state
    elif mode == "full":
        prompt = torch.cat((context.flatten(), prompt))
    scores = []
    token_argmax = False
    for i, candidate in enumerate(candidates):
        if adapter is not None:
            adapter.state = prefix_state
        tokens = torch.tensor(candidate, device=device)
        ids = torch.cat((prompt, tokens[:-1]))
        logits = lm(input_ids=ids[None], use_cache=False, logits_to_keep=len(candidate)).logits[0].float()
        logps = logits.log_softmax(dim=-1)
        scores.append(float(logps.gather(1, tokens[:, None]).sum().item()))
        if i == 0:
            token_argmax = bool((logits.argmax(dim=-1) == tokens).all())
    if not np.isfinite(scores).all():
        raise FloatingPointError(f"Non-finite candidate score for {mode}")
    # Treat ties conservatively.
    return scores, 1 + sum(score >= scores[0] for score in scores[1:]), token_argmax


output_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
trials = []
for distance in DISTANCES:
    for seed in SEEDS:
        for trial in range(TRIALS_PER_SEED):
            clean, noisy, prompt, candidates, values = make_trial(seed, trial, distance)
            window = score_candidates("window", clean, prompt, candidates)
            for condition, context in (("clean", clean), ("noisy", noisy)):
                for mode in MODELS:
                    scores, rank, token_argmax = window if mode == "window" else score_candidates(mode, context, prompt, candidates)
                    trials.append(dict(distance=distance, seed=seed, trial=trial, condition=condition,
                                       mode=mode, context_tokens=context.numel(), candidates=values,
                                       logps=scores, rank=rank, token_argmax=token_argmax))
    print(f"Distance {distance}: {len(SEEDS) * TRIALS_PER_SEED} pairs", flush=True)
    (output_dir / "noisy_trials.json").write_text(json.dumps(trials, ensure_ascii=False), encoding="utf-8")


def interval(values, draws):
    samples = values[draws].mean(axis=1)
    return [float(values.mean()), *map(float, np.quantile(samples, [0.025, 0.975]))]


summary = []
bootstrap_rng = np.random.default_rng(2026)
for distance in DISTANCES:
    for mode in MODELS:
        groups = {}
        for condition in ("clean", "noisy"):
            batch = [t for t in trials if (t["distance"], t["mode"], t["condition"]) == (distance, mode, condition)]
            groups[condition] = np.array([[t["logps"][0], t["rank"] == 1, 1 / t["rank"], t["token_argmax"]] for t in batch])
        draws = bootstrap_rng.integers(len(batch), size=(5000, len(batch)))
        for condition, measurements in (*groups.items(), ("noisy_minus_clean", groups["noisy"] - groups["clean"])):
            summary.append(dict(distance=distance, mode=mode, condition=condition,
                                **{metric: interval(measurements[:, i], draws)
                                   for i, metric in enumerate(("logp", "rank1", "mrr", "teacher_forced_token_argmax"))}))

report = dict(train_steps={k: m.train_steps for k, m in adapters.items()}, seeds=SEEDS,
              trials_per_seed=TRIALS_PER_SEED, distractors=3, summary=summary)
(output_dir / "noisy_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
with (output_dir / "noisy_summary.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["distance", "mode", "condition", "metric", "mean", "ci_low", "ci_high"])
    for row in summary:
        for metric in ("logp", "rank1", "mrr", "teacher_forced_token_argmax"):
            writer.writerow([row["distance"], row["mode"], row["condition"], metric, *row[metric]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, condition in zip(axes, ("clean", "noisy")):
    for mode in MODELS:
        means = [row["rank1"][0] for row in summary if row["mode"] == mode and row["condition"] == condition]
        ax.plot(DISTANCES, means, marker="o", label=PRETTY.get(mode, mode))
    ax.set(title=condition, xlabel="Segments after the target segment", ylabel="Top-1 of four candidates", ylim=(0, 1.05))
    ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "noisy_rank1.png", dpi=150)
